# Feature Engineering en SQL

A continuación, veremos cómo calcular diferentes variables para el feature engineering utilizando SQL.


In [16]:
%pip install jupysql
%pip install duckdb-engine

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [17]:
import duckdb
import pandas as pd

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

%sql duckdb://

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [18]:
from pathlib import Path

dataset_dir = Path('/Users/thenseler/Documents/Maestria/DMEyF/dmeyf2026/data')
dataset_file = 'competencia_01.csv'
dataset_file_path = dataset_dir / dataset_file
if not dataset_file_path.is_file():
    raise FileNotFoundError(f'No se encontró el dataset local: {dataset_file_path}')
dataset_path = dataset_dir.as_posix() + '/'
print(f'Cargando dataset local: {dataset_file_path}')

Cargando dataset local: /Users/thenseler/Documents/Maestria/DMEyF/dmeyf2026/data/competencia_01.csv


In [19]:
%%sql
create or replace table competencia_01 as
select
    *
from read_csv_auto("{{dataset_path + dataset_file}}")

,Success


In [20]:
%%sql
select
    Master_Fvencimiento
    , Visa_Fvencimiento
    , greatest(Master_Fvencimiento, Visa_Fvencimiento) as tc_fvencimiento_mayor
    , least(Master_Fvencimiento, Visa_Fvencimiento) as tc_fvencimiento_menor
from competencia_01 limit 10

,Master_Fvencimiento,Visa_Fvencimiento,tc_fvencimiento_mayor,tc_fvencimiento_menor
0,-1794,-1794,-1794,-1794
1,-1763,-1763,-1763,-1763
2,-2190,-2190,-2190,-2190
3,-1035,-1401,-1035,-1401
4,-1005,-1371,-1005,-1371
5,-974,-1340,-974,-1340
6,-944,-1310,-944,-1310
7,-913,-1279,-913,-1279
8,-882,-1248,-882,-1248
9,-2039,-2039,-2039,-2039


Lo siguiente es querer operar dos variables, como por ejemplo sumarla. Esto es sencillo


In [21]:
%%sql
select
    Master_msaldototal
    , Visa_msaldototal
    , Master_msaldototal + Visa_msaldototal as tc_saldo_total
from competencia_01 limit 10

,Master_msaldototal,Visa_msaldototal,tc_saldo_total
0,0.00,17350.75,17350.75
1,0.00,30528.48,30528.48
2,0.00,0.00,0.00
3,857.26,17293.33,18150.59
4,857.06,17772.51,18629.57
5,856.86,16260.89,17117.75
6,0.00,20830.47,20830.47
7,0.00,50631.87,50631.87
8,0.00,46703.76,46703.76
9,0.00,0.00,0.00


Pero un DS de a de veras mirará los datos y se encontrará con un campo que es null cuando se lo suma a otro dará null.

In [22]:
%%sql
select
    Master_msaldototal
    , Visa_msaldototal
    , Master_msaldototal + Visa_msaldototal as tc_saldo_total
from competencia_01 where Master_msaldototal is null limit 10

,Master_msaldototal,Visa_msaldototal,tc_saldo_total
0,NaN,89335.11,NaN
1,NaN,34497.59,NaN
2,NaN,44186.66,NaN
3,NaN,53329.13,NaN
4,NaN,77392.77,NaN
5,NaN,NaN,NaN
6,NaN,NaN,NaN
7,NaN,NaN,NaN
8,NaN,NaN,NaN
9,NaN,NaN,NaN


Esto no siempre es deseable y puede ser fácilmente evitable

In [23]:
%%sql
select
    Master_msaldototal
    , Visa_msaldototal
    , ifnull(Master_msaldototal, 0) + ifnull(Visa_msaldototal, 0) as tc_saldo_total
from competencia_01 limit 10

,Master_msaldototal,Visa_msaldototal,tc_saldo_total
0,0.00,17350.75,17350.75
1,0.00,30528.48,30528.48
2,0.00,0.00,0.00
3,857.26,17293.33,18150.59
4,857.06,17772.51,18629.57
5,856.86,16260.89,17117.75
6,0.00,20830.47,20830.47
7,0.00,50631.87,50631.87
8,0.00,46703.76,46703.76
9,0.00,0.00,0.00


In [24]:
%%sql
CREATE OR REPLACE MACRO suma_sin_null(a, b) AS ifnull(a, 0) + ifnull(b, 0);


,Success


In [25]:
%%sql
select distinct
    Master_msaldototal
    , Visa_msaldototal
    , suma_sin_null(Master_msaldototal, Visa_msaldototal) as tc_saldo_total
from competencia_01 where Master_msaldototal is null limit 10


,Master_msaldototal,Visa_msaldototal,tc_saldo_total
0,NaN,33731.73,33731.73
1,NaN,26842.48,26842.48
2,NaN,NaN,0.00
3,NaN,4626.72,4626.72
4,NaN,16012.84,16012.84
5,NaN,15713.67,15713.67
6,NaN,80787.38,80787.38
7,NaN,16297.41,16297.41
8,NaN,16112.84,16112.84
9,NaN,29207.53,29207.53


TAREA: Escriba una macro para hacer un ratio de dos variables que sea seguro, donde no solo hay campos con null, también esta el problema de la división por cero. Como es costumbre comparta su solución por este canal. Lea https://duckdb.org/docs/sql/functions/numeric.html para referencias de funciones que puede usar.

---

"Claro!" me dirá, mientras lee esto con un mate en la mano, "para cosas fáciles usar SQL alcanza, pero para algo más complicado como crear campos contra el data drifting es difícil".... elija su medicina:

In [26]:
%%sql
select
    foto_mes
    , numero_de_cliente
    , cliente_antiguedad
    , row_number() over (partition by numero_de_cliente order by foto_mes) as cliente_antiguedad_2
    , percent_rank() over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_3
    , cume_dist() over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_4
    , ntile(4) over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_5
    , ntile(10) over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_6
from competencia_01
order by numero_de_cliente, cliente_antiguedad


,foto_mes,numero_de_cliente,cliente_antiguedad,cliente_antiguedad_2,cliente_antiguedad_3,cliente_antiguedad_4,cliente_antiguedad_5,cliente_antiguedad_6
0,202103,12159854,134,1,0.545491,0.550712,3,6
1,202104,12159854,135,2,0.547454,0.552706,3,6
2,202105,12159854,136,3,0.549647,0.554852,3,6
3,202106,12159854,137,4,0.551510,0.556704,3,6
4,202107,12159854,138,5,0.553128,0.558303,3,6
...,...,...,...,...,...,...,...,...
983056,202108,78249586,1,1,0.000000,0.001585,1,1
983057,202108,78249850,1,1,0.000000,0.001585,1,1
983058,202108,78250419,1,1,0.000000,0.001585,1,1
983059,202108,78253625,1,1,0.000000,0.001585,1,1


Qué paso? use las hermosas funciones analíticas de SQL. Al campo cliente_antiguedad (que no sufre de data drifting, solo esta para dar el ejemplo) para cada período (partition by foto_mes) la ordeno (order by cliente_antiguedad) y luego calculo las métricas de orden que pueden encontrar acá https://duckdb.org/docs/sql/window_functions.html#general-purpose-window-functions.

Seguiremos usando las funciones analíticas de SQL, esta vez para calcular features que utilizan valores del pasado.

Qué pasa si quiero agregar un feature que muestre el valor del periodo anterior?


In [27]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , lag(ctrx_quarter, 1) over (partition by numero_de_cliente order by foto_mes) as lag_1_ctrx_quarter
from competencia_01
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,lag_1_ctrx_quarter
0,17046283,202106,46,38
1,17046283,202107,42,46
2,17046283,202108,42,42
3,17049973,202103,1,<NA>
4,17049973,202104,1,1
5,17049973,202105,1,1
6,17049973,202106,0,1
7,17049973,202107,0,0
8,17049973,202108,0,0
9,17052066,202103,98,<NA>


Podemos calcular el delta (diferencia) entre el valor pasado y el presente, para uno o varios meses


In [28]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , lag(ctrx_quarter, 1) over (partition by numero_de_cliente order by foto_mes) as lag_1_ctrx_quarter
  , ctrx_quarter - lag_1_ctrx_quarter as delta_1_ctrx_quarter
  , ctrx_quarter - lag(ctrx_quarter, 2) over (partition by numero_de_cliente order by foto_mes) as lag_2_ctrx_quarter
from competencia_01
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,lag_1_ctrx_quarter,delta_1_ctrx_quarter,lag_2_ctrx_quarter
0,17046283,202106,46,38,8,10
1,17046283,202107,42,46,-4,4
2,17046283,202108,42,42,0,-4
3,17049973,202103,1,<NA>,<NA>,<NA>
4,17049973,202104,1,1,0,<NA>
5,17049973,202105,1,1,0,0
6,17049973,202106,0,1,-1,-1
7,17049973,202107,0,0,0,-1
8,17049973,202108,0,0,0,0
9,17052066,202103,98,<NA>,<NA>,<NA>


Si necesitamos ya no solo traer un valor del pasado, sino una secuencia de valores, por ejemplo para calcular la media móvil con los últimos 3 meses anteriores? se puede hacer fácilmente


In [29]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , lag(ctrx_quarter, 1) over (partition by numero_de_cliente order by foto_mes) as lag_1_ctrx_quarter
  , lag(ctrx_quarter, 2) over (partition by numero_de_cliente order by foto_mes) as lag_2_ctrx_quarter
  , lag(ctrx_quarter, 3) over (partition by numero_de_cliente order by foto_mes) as lag_3_ctrx_quarter
  , avg(ctrx_quarter) over (partition by numero_de_cliente
                            order by foto_mes
                            rows between 3 preceding and current row) as avg_3_ctrx_quarter
from competencia_01
order by numero_de_cliente, foto_mes desc
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,lag_1_ctrx_quarter,lag_2_ctrx_quarter,lag_3_ctrx_quarter,avg_3_ctrx_quarter
0,12159854,202108,43,45,44,54,46.500000
1,12159854,202107,45,44,54,59,50.500000
2,12159854,202106,44,54,59,64,55.250000
3,12159854,202105,54,59,64,<NA>,59.000000
4,12159854,202104,59,64,<NA>,<NA>,61.500000
5,12159854,202103,64,<NA>,<NA>,<NA>,64.000000
6,12159858,202108,70,68,60,64,65.500000
7,12159858,202107,68,60,64,71,65.750000
8,12159858,202106,60,64,71,65,65.000000
9,12159858,202105,64,71,65,<NA>,66.666667


Si embargo puede resultar incómodo escribir constantemente el over partition sobre todo si se buscan aplicar muchas veces para distintas funciones. Para reducir el código se puede usar la siguiente sintaxis



In [30]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , avg(ctrx_quarter) over ventana_3 as ctrx_quarter_media_3
  , max(ctrx_quarter) over ventana_3 as ctrx_quarter_max_3
  , min(ctrx_quarter) over ventana_3 as ctrx_quarter_min_3
from competencia_01
window ventana_3 as (partition by numero_de_cliente order by foto_mes rows between 3 preceding and current row)
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,ctrx_quarter_media_3,ctrx_quarter_max_3,ctrx_quarter_min_3
0,12160484,202103,65,65.000000,65,65
1,12160484,202104,52,58.500000,65,52
2,12160484,202105,40,52.333333,65,40
3,12160484,202106,33,47.500000,65,33
4,12160484,202107,28,38.250000,52,28
5,12160484,202108,35,34.000000,40,28
6,12160968,202103,60,60.000000,60,60
7,12160968,202104,53,56.500000,60,53
8,12160968,202105,61,58.000000,61,53
9,12160968,202106,58,58.000000,61,53


Para saber más que funciones tenemos disponibles, recomiendo ver los siguientes links:

https://duckdb.org/docs/archive/0.8.1/sql/window_functions
https://duckdb.org/docs/archive/0.8.1/sql/aggregates
Un caso más, que ni me voy a molestar en explicar que significa...


In [31]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  ,regr_slope(ctrx_quarter, cliente_antiguedad) over ventana_3 as ctrx_quarter_slope_3
from competencia_01
window ventana_3 as (partition by numero_de_cliente order by foto_mes rows between 3 preceding and current row)
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,ctrx_quarter_slope_3
0,12160484,202103,65,NaN
1,12160484,202104,52,-13.0
2,12160484,202105,40,-12.5
3,12160484,202106,33,-10.8
4,12160484,202107,28,-7.9
5,12160484,202108,35,-2.0
6,12160968,202103,60,NaN
7,12160968,202104,53,-7.0
8,12160968,202105,61,0.5
9,12160968,202106,58,0.2


... Alguno dirá "tenemos que escribir todo esto a mano? Son muchas variables!". Bueno no, use los conocimientos de programación para que la computadora trabaje para usted. Si tenemos una lista de campos


In [32]:
campos = ['active_quarter', 'cliente_vip', 'internet', 'cliente_edad', 'cliente_antiguedad', 'mrentabilidad']


Podemos hacer un script muy sencillo que nos genere el texto que hay que poner en una query para generar esas variables


In [33]:
nuevos_features = ""
for campo in campos:
  nuevos_features += f"\n, regr_slope({campo}, cliente_antiguedad) over ventana_3 as ctrx_{campo}_slope_3"
print(nuevos_features)



, regr_slope(active_quarter, cliente_antiguedad) over ventana_3 as ctrx_active_quarter_slope_3
, regr_slope(cliente_vip, cliente_antiguedad) over ventana_3 as ctrx_cliente_vip_slope_3
, regr_slope(internet, cliente_antiguedad) over ventana_3 as ctrx_internet_slope_3
, regr_slope(cliente_edad, cliente_antiguedad) over ventana_3 as ctrx_cliente_edad_slope_3
, regr_slope(cliente_antiguedad, cliente_antiguedad) over ventana_3 as ctrx_cliente_antiguedad_slope_3
, regr_slope(mrentabilidad, cliente_antiguedad) over ventana_3 as ctrx_mrentabilidad_slope_3





Con la salida de esa celda, arme la query agregando las nuevas líneas y la ejecuta.

Lo que acabamos de hacer de manera muy simple es como "funcionan" sistemas como **dbt** que están tan de moda en el mundo de los datos.

La última reflexión, la creación de nuevas features es un proceso computacionalmente rápido pero intenso. Si ejecutó lo anterior pudo haber visto que en poco minutos tenía sus nuevas variables. Pero, también pudo haberle fallado por temas de recursos. Miles de variables necesitan los recursos adecuados. Use la nube, una máquina grande, al menos que sepa bien como optimizar las queries.


Y a no olvidarse guardar las nueva tabla

In [34]:
%%sql
COPY competencia_01 TO '{dataset_path}competencia_01_fe.csv' (FORMAT CSV, HEADER TRUE);


,Success
